# Spark ETL — flights from Kafka to results

This notebook is the **core processing stage** of the pipeline:

`Kafka (group10.flights) -> Spark -> filter commercial -> join reference data -> aggregate -> parquet`

It answers two of the project questions:
- **Q1 Dominance** — which airlines / carrier nations fly the most over Europe?
- **Q3 Busiest airports** — inferred by matching low climbing/descending aircraft to their nearest airport (a spatial join).

**Run this on the JupyterHub** (it must reach the Spark cluster). It reads **live** from Kafka, so make sure the producer has run recently / is running:
```
python -m src.producer            # continuous
python -m src.producer --once     # single snapshot
```
Note: the shared broker has short retention, so Kafka only holds the most recent data — that is exactly why we persist results to parquet here.

## 1. SparkSession
We connect to the cluster master and pull in the Kafka connector package (`spark-sql-kafka`, matched to the cluster's Spark 3.5.6). On a fresh kernel the first run downloads the connector jars via Ivy (one-time). `setLogLevel("ERROR")` quiets the routine WARN noise.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("group10-flights-etl")
    .master("spark://172.29.16.102:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| app", spark.sparkContext.applicationId)

## 2. Read from Kafka and parse the JSON
Each Kafka message is one aircraft as a JSON string. We batch-read the whole topic (`startingOffsets=earliest`) and parse the JSON into typed columns using an explicit schema.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType, LongType,
)

schema = StructType([
    StructField("icao24", StringType()),
    StructField("callsign", StringType()),
    StructField("origin_country", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("baro_altitude", DoubleType()),
    StructField("on_ground", BooleanType()),
    StructField("velocity", DoubleType()),
    StructField("true_track", DoubleType()),
    StructField("vertical_rate", DoubleType()),
    StructField("geo_altitude", DoubleType()),
    StructField("snapshot_time", LongType()),
])

raw = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "172.29.16.101:9092")
    .option("subscribe", "group10.flights")
    .option("startingOffsets", "earliest")
    .load())

flights = raw.select(F.from_json(F.col("value").cast("string"), schema).alias("f")).select("f.*")
print("parsed flights:", flights.count())

## 3. Clean
Keep only airborne aircraft with a callsign, and derive `airline_icao` = the first 3 characters of the callsign (the airline's ICAO code).

In [ ]:
clean = (flights
    .filter(~F.col("on_ground"))
    .filter(F.col("callsign") != "")
    .withColumn("airline_icao", F.substring("callsign", 1, 3)))
print("airborne w/ callsign:", clean.count())

## 4. Keep only commercial flights (broadcast join with the airline reference)
We load the scraped airline-codes lookup (Wikipedia, source 2) straight from the repo on GitHub, broadcast it (it's small), and **inner-join** on the ICAO code. The inner join doubles as the **commercial filter**: private/military/GA callsigns have no airline match and drop out.

In [ ]:
import pandas as pd

AIRLINES_CSV = "https://raw.githubusercontent.com/jakobiene/bde-project-group10/main/data/processed/airlines.csv"
airlines = spark.createDataFrame(pd.read_csv(AIRLINES_CSV))

commercial = clean.join(F.broadcast(airlines), clean.airline_icao == airlines.icao, "inner")
print("commercial flights:", commercial.count())

## 5. Q1 — Dominance
Distinct aircraft per airline and per operator country.

In [ ]:
by_airline = (commercial.groupBy("airline_icao", "airline", "country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))

by_country = (commercial.groupBy("origin_country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))

by_airline.show(10, truncate=False)
by_country.show(10, truncate=False)

## 6. Store Q1 results (ETL output)
Results are small, so we collect to the driver and write parquet locally (this avoids distributed-write issues on the shared cluster). If `to_parquet` lacks an engine, switch to `.to_csv(...)`.

In [ ]:
import os
OUT_DIR = "data/processed"
os.makedirs(OUT_DIR, exist_ok=True)
by_airline.toPandas().to_parquet(f"{OUT_DIR}/dominance_by_airline.parquet", index=False)
by_country.toPandas().to_parquet(f"{OUT_DIR}/dominance_by_country.parquet", index=False)
print("Q1 results written to", os.path.abspath(OUT_DIR))

## 7. Q3 — Busiest airports (spatial join)
Load the airports reference (OurAirports, source 3 — European commercial airports with coordinates), then pick **arrival/departure candidates**: aircraft that are **low (<3000 m) and climbing/descending** (`|vertical_rate| > 1`). Such aircraft must be near an airport (this also drops the negative-altitude outliers).

In [ ]:
AIRPORTS_CSV = "https://raw.githubusercontent.com/jakobiene/bde-project-group10/main/data/processed/airports.csv"
airports = spark.createDataFrame(pd.read_csv(AIRPORTS_CSV))
print("airports:", airports.count())

arrdep = (commercial
    .filter(F.col("geo_altitude").isNotNull())
    .filter((F.col("geo_altitude") >= 0) & (F.col("geo_altitude") < 3000))
    .filter(F.abs(F.col("vertical_rate")) > 1)
    .select("icao24", "callsign", "latitude", "longitude", "vertical_rate", "geo_altitude"))
print("arrival/departure candidates:", arrdep.count())

For each candidate we cross-join all airports, compute the **haversine distance**, keep the **nearest** airport, and require it to be within **30 km** (so the aircraft is genuinely at that airport). Counting distinct aircraft per airport gives the busiest-airports ranking.

In [ ]:
from pyspark.sql.window import Window
R = 6371.0  # earth radius, km

ap = airports.select(
    F.col("icao_code"), F.col("name").alias("airport_name"),
    F.col("iso_country"), F.col("latitude_deg"), F.col("longitude_deg"))

pairs = arrdep.crossJoin(F.broadcast(ap)).withColumn(
    "dist_km",
    2 * R * F.asin(F.sqrt(
        F.pow(F.sin(F.radians(F.col("latitude_deg") - F.col("latitude")) / 2), 2)
        + F.cos(F.radians(F.col("latitude"))) * F.cos(F.radians(F.col("latitude_deg")))
        * F.pow(F.sin(F.radians(F.col("longitude_deg") - F.col("longitude")) / 2), 2)
    )))

w = Window.partitionBy("icao24").orderBy("dist_km")
nearest = (pairs.withColumn("rk", F.row_number().over(w))
    .filter(F.col("rk") == 1)
    .filter(F.col("dist_km") <= 30))

busiest = (nearest.groupBy("icao_code", "airport_name", "iso_country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))
busiest.show(15, truncate=False)

In [ ]:
busiest.toPandas().to_parquet(f"{OUT_DIR}/busiest_airports.parquet", index=False)
print("Q3 results written to", os.path.abspath(OUT_DIR))

## Summary
This notebook demonstrated the required **Kafka -> Spark -> storage** ETL:
- read the live `group10.flights` stream from Kafka,
- cleaned it and filtered to **commercial** flights via a broadcast join with the scraped airline codes,
- produced **Q1 (dominance)** and **Q3 (busiest airports, spatial join)**,
- stored all results to parquet in `data/processed/`.

Because the Kafka broker only retains data briefly, a single run reflects one moment in time. For robust results we accumulate many snapshots into `flights_history.parquet` (see the collector) and run the analysis/visualisation over that history.